# FarmTech Solutions — Entrega 1 (Fase 6)
**Aluno:** Diego Filipe Pereira de Araujo — RM567064

**Ambiente:** **Google Colab** (GPU recomendada: *Runtime → Change runtime type → T4*). Este notebook **não** usa paths locais do PC.

**Objetivo:** pipeline **YOLOv5** para duas classes (`objeto_a`, `objeto_b`), treino, validação e teste, comparando **30** e **60 épocas**.


## 1. Dataset (enunciado FIAP)

- **80 imagens**, split 32/4/4 por classe; pasta **`dataset_yolo_farmtech`** com `train/`, `valid/`, `test/`.
- **No Colab:** compacte a pasta do seu PC em **`dataset_yolo_farmtech.zip`** (a raiz do zip deve conter a pasta `dataset_yolo_farmtech/` com `train`, `valid`, `test`). Envie o zip pelo menu **Arquivos** (ícone de pasta) para **`/content/farmtech/`** e rode a célula de setup abaixo (ela descompacta e clona o YOLO).
- **Alternativa:** deixe o projeto no **Google Drive**, monte o Drive na célula de setup e ajuste `PROJECT` para o caminho da pasta **Capítulo 1** (a que contém `assets/dataset_yolo_farmtech`).
- **Make Sense IA:** export YOLO. A célula de rótulos **não sobrescreve** `.txt` já exportados.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

# ========== Google Colab: caminho do projeto ==========
# Padrão: tudo fica em /content/farmtech
PROJECT = Path("/content/farmtech")
USE_GOOGLE_DRIVE = False
# Se True, ajuste DRIVE_PROJECT para a pasta do repositório que contém assets/dataset_yolo_farmtech
DRIVE_PROJECT = Path("/content/drive/MyDrive/SEU_CAMINHO/FIAP-ON-AI/Fase 6/Capitulo 1")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = DRIVE_PROJECT

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "assets").mkdir(parents=True, exist_ok=True)

DATA_ROOT = PROJECT / "assets" / "dataset_yolo_farmtech"
ZIP_ALVO = PROJECT / "dataset_yolo_farmtech.zip"

if ZIP_ALVO.is_file() and not DATA_ROOT.is_dir():
    with zipfile.ZipFile(ZIP_ALVO, "r") as zf:
        zf.extractall(PROJECT / "assets")
    print("Descompactado:", ZIP_ALVO, "->", PROJECT / "assets")

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        f"Dataset não encontrado: {DATA_ROOT}\n"
        "Envie dataset_yolo_farmtech.zip para /content/farmtech/ (Menu Arquivos) "
        "ou defina USE_GOOGLE_DRIVE=True e DRIVE_PROJECT com o caminho correto."
    )

YOLO_DIR = PROJECT / "yolov5"
YAML_PATH = PROJECT / "data_fase6.yaml"
CLASSES = ["objeto_a", "objeto_b"]

if not YOLO_DIR.is_dir():
    subprocess.run(
        ["git", "clone", "https://github.com/ultralytics/yolov5.git", str(YOLO_DIR)],
        check=True,
        cwd=str(PROJECT),
    )

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(YOLO_DIR / "requirements.txt")],
    check=True,
)
(PROJECT / "runs").mkdir(parents=True, exist_ok=True)
os.chdir(YOLO_DIR)
print("PROJECT:", PROJECT)
print("DATA_ROOT:", DATA_ROOT)
print("YOLO_DIR:", YOLO_DIR)
print("YAML (raiz do projeto):", YAML_PATH)
print("Saídas de treino/detect:", PROJECT / "runs")
print("Python:", sys.executable)


In [ ]:
from pathlib import Path


def ensure_yolo_labels(images_dir: Path, labels_dir: Path) -> tuple[int, int]:
    """Gera .txt só para imagens sem rótulo (preserva export do Make Sense)."""
    labels_dir.mkdir(parents=True, exist_ok=True)
    criados = 0
    total = 0
    for img in sorted(images_dir.glob("*")):
        if img.suffix.lower() not in (".jpg", ".jpeg", ".png", ".webp", ".bmp"):
            continue
        stem = img.stem
        if stem.startswith("objeto_a"):
            classe = 0
        elif stem.startswith("objeto_b"):
            classe = 1
        else:
            continue
        total += 1
        lbl = labels_dir / (stem + ".txt")
        if lbl.is_file():
            continue
        linha = f"{classe} 0.5 0.5 0.82 0.82\n"
        lbl.write_text(linha, encoding="utf-8")
        criados += 1
    return criados, total


assert DATA_ROOT.is_dir(), f"Dataset não encontrado: {DATA_ROOT}"

for split in ("train", "valid"):
    cria, tot = ensure_yolo_labels(DATA_ROOT / split / "images", DATA_ROOT / split / "labels")
    print(f"{split}: {tot} imagens com classe no nome; {cria} rótulos novos (faltavam).")

# Caminhos relativos à raiz do projeto (onde está data_fase6.yaml)
yaml_text = f"""train: assets/dataset_yolo_farmtech/train/images
val: assets/dataset_yolo_farmtech/valid/images
test: assets/dataset_yolo_farmtech/test/images

nc: {len(CLASSES)}
names: {CLASSES}
"""
YAML_PATH.write_text(yaml_text, encoding="utf-8")
print(YAML_PATH.read_text())


## 2. Treinamento — experimento 1 (**30 épocas**)
Modelo base **YOLOv5s** pré-treinado no COCO, ajustado ao nosso `data_fase6.yaml`. Batch 16 e imagem 640 — equilíbrio entre tempo e qualidade no Colab.


In [ ]:
# Treino 30 épocas — saída em PROJECT/runs/train/fase6_ep30 (cwd = yolov5)
!WANDB_MODE=disabled python3 train.py --img 640 --batch 16 --epochs 30 --data ../data_fase6.yaml --weights yolov5s.pt --project ../runs/train --name fase6_ep30


## 3. Treinamento — experimento 2 (**60 épocas**)
Mesma configuração, mudando só o número de épocas para comparar convergência e tempo total.


In [ ]:
# Treino 60 épocas — PROJECT/runs/train/fase6_ep60
!WANDB_MODE=disabled python3 train.py --img 640 --batch 16 --epochs 60 --data ../data_fase6.yaml --weights yolov5s.pt --project ../runs/train --name fase6_ep60


## 4. Validação
Mede **mAP** e perdas no conjunto de **validação** definido no YAML (não é o teste). Rodamos os dois `best.pt`.


In [ ]:
# Validação dos dois experimentos
!WANDB_MODE=disabled python3 val.py --weights ../runs/train/fase6_ep30/weights/best.pt --data ../data_fase6.yaml --img 640
!WANDB_MODE=disabled python3 val.py --weights ../runs/train/fase6_ep60/weights/best.pt --data ../data_fase6.yaml --img 640

## 5. Teste — inferência
Rodamos **detect** nas imagens de **test/images** (sem rótulo no fluxo típico). Saída em `runs/detect/teste_ep30` e `runs/detect/teste_ep60`. Essas imagens servem para mostrar o modelo ao “cliente” e para prints da entrega.


In [ ]:
# Inferência no conjunto de teste
!WANDB_MODE=disabled python3 detect.py --weights ../runs/train/fase6_ep30/weights/best.pt --img 640 --conf 0.25 --source {DATA_ROOT/'test/images'} --project ../runs/detect --name teste_ep30 --exist-ok
!WANDB_MODE=disabled python3 detect.py --weights ../runs/train/fase6_ep60/weights/best.pt --img 640 --conf 0.25 --source {DATA_ROOT/'test/images'} --project ../runs/detect --name teste_ep60 --exist-ok

## 6. Comparação numérica (30 × 60 épocas)
Leitura da última linha de `results.csv` de cada treino: **mAP@0.5**, **mAP@0.5:0.95** e perdas de treino (referência de **erro** e **desempenho**). Em detecção, o analog à “acurácia” costuma ser o **mAP** e a precisão/recall impressos no `val`.


In [ ]:
import pandas as pd


def last_metrics(path: str):
    df = pd.read_csv(path)
    row = df.iloc[-1]
    keys = {k.strip(): k for k in df.columns}

    def pegar(nome):
        for cand in (nome, " " + nome, " " + nome.strip()):
            k = keys.get(cand.strip())
            if k is not None:
                return row[k]
        return None

    return {
        "mAP_0.5": pegar("metrics/mAP_0.5") or pegar("metrics/mAP50"),
        "mAP_0.5:0.95": pegar("metrics/mAP_0.5:0.95") or pegar("metrics/mAP50-95"),
        "train/obj_loss": pegar("train/obj_loss"),
        "train/box_loss": pegar("train/box_loss"),
    }


comparacao = pd.DataFrame([
    {"experimento": "30 épocas", **last_metrics("../runs/train/fase6_ep30/results.csv")},
    {"experimento": "60 épocas", **last_metrics("../runs/train/fase6_ep60/results.csv")},
])
comparacao

## 7. Conclusões (validação e teste)

**Comparação 30 × 60 épocas:** Em geral, **mais épocas** tendem a reduzir perda e melhorar **mAP**, até aparecer **overfitting** ou ganhos pequenos pelo tempo extra. Compare na tabela acima qual experimento teve **mAP@0.5** maior.

**Validação:** o `val.py` resume quão bem o modelo generaliza para imagens **parecidas** com treino, mas **fora** do treino.

**Teste:** o `detect` mostra caixas nas fotos novas (pasta `test`). É o melhor critério visual para o cenário FarmTech.

**Pontos fortes:** pipeline completo YOLO; dataset balanceado 40/40; duas configurações de épocas comparáveis.

**Limitações:** poucas imagens por split; objetos simples (frutas); sensível a iluminação e fundo; caixas devem ser bem feitas no Make Sense — rótulos ruins pioram muito o resultado.


In [ ]:
# Exibe alguns prints da inferência no teste (experimentos 30 e 60)
from pathlib import Path
from IPython.display import display, Image

def mostrar_run(nome_pasta: str, max_img: int = 4):
    pasta = Path("../runs/detect") / nome_pasta
    if not pasta.is_dir():
        print(f"Pasta não encontrada: {pasta}")
        return
    imgs = [p for p in sorted(pasta.glob("*")) if p.suffix.lower() in (".jpg", ".jpeg", ".png")]
    print(f"=== {nome_pasta} ({len(imgs)} arquivos) ===")
    for p in imgs[:max_img]:
        display(Image(filename=str(p)))

mostrar_run("teste_ep30")
mostrar_run("teste_ep60")

## 8. Prints para o GitHub / vídeo
Faça **download** das pastas `runs/detect/teste_ep60/` ou `teste_ep30/` pelo menu **Arquivos** do Colab (botão direito → Download). No PC, copie imagens representativas para `assets/exemplos_teste/` no repositório Git e cite no **README** e no vídeo (YouTube não listado).
